# Source Layer Ingestion

Purpose:

Create governed Source Delta tables from
the TEP raw source assets stored in Unity Catalog Volumes.

Responsibilities:

- Read source files
- Validate contracts
- Publish Source tables

No feature engineering occurs here.

In [0]:
# check if pandas is installed
import pandas as pd
print(pd.__version__)

In [0]:
# check if pyreadr installed for reading RData files
import pyreadr
print(pyreadr.__version__)

In [0]:
display(dbutils.fs.ls("/Volumes/tep_anomaly/source/raw_files"))

## Convert faultfree_training.parquet to delta table

In [0]:
# display fault_free_training
faultfree_training_df = spark.read.parquet(
    "dbfs:/Volumes/tep_anomaly/source/raw_files/fault_free_training.parquet"
)

display(faultfree_training_df.limit(10))

In [0]:
# data contract validation
required_columns = [
    "faultNumber",
    "simulationRun",
    "sample"
]

missing_columns = [
    col for col in required_columns
    if col not in faultfree_training_df.columns
]

if missing_columns:
    raise Exception(
        f"Missing required columns: {missing_columns}"
    )

print("Data contract validation passed.")


In [0]:
# write it as a delta tables
(
    faultfree_training_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "tep_anomaly.source.faultfree_training"
    )
)

In [0]:
# verify delta table creation
display(
    spark.sql("""
    SELECT COUNT(*)
    FROM tep_anomaly.source.faultfree_training
    """)
)

## Source ingestion workflow

In [0]:
from src.ingestion.source_ingestion import (
    validate_required_columns,
    validate_row_count
)

## Fault Free Testing

In [0]:
# read faulfree_testing
faultfree_testing_df = spark.read.parquet(
    "dbfs:/Volumes/tep_anomaly/source/raw_files/fault_free_testing.parquet"
)

# data_contract_validation
validate_required_columns(
    faultfree_testing_df,
    ["faultNumber", "simulationRun", "sample"]
)

# validate row count
validate_row_count(faultfree_testing_df)
(
    faultfree_testing_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "tep_anomaly.source.faultfree_testing"
    )
)

## Faulty Training


In [0]:
# read faulfree_testing
faulty_training_df = spark.read.parquet(
    "dbfs:/Volumes/tep_anomaly/source/raw_files/faulty_training.parquet"
)

# data_contract_validation
validate_required_columns(
    faulty_training_df,
    ["faultNumber", "simulationRun", "sample"]
)

# validate row count
validate_row_count(faulty_training_df)
(
    faulty_training_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "tep_anomaly.source.faulty_training"
    )
)

In [0]:
# read faulfree_testing
faulty_testing_df = spark.read.parquet(
    "dbfs:/Volumes/tep_anomaly/source/raw_files/faulty_testing.parquet"
)

# data_contract_validation
validate_required_columns(
    faulty_testing_df,
    ["faultNumber", "simulationRun", "sample"]
)

# validate row count
validate_row_count(faulty_testing_df)
(
    faulty_testing_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "tep_anomaly.source.faulty_testing"
    )
)

## Verify all tables

In [0]:
%sql
SELECT COUNT(*) FROM tep_anomaly.source.faultfree_training;

In [0]:
%sql
SELECT COUNT(*) FROM tep_anomaly.source.faultfree_testing;

In [0]:
%sql
SELECT COUNT(*) FROM tep_anomaly.source.faulty_training;

In [0]:
%sql
SELECT COUNT(*) FROM tep_anomaly.source.faulty_testing;